In [5]:
import pickle
from elasticsearch import Elasticsearch
import os

from workers.utils.task_definitions import (
    TermDistribution,
    WordCloud,
    RelatedTermCounts,
    RelationFinder,
    DocsRelatedFinder
)

INDEX_NAME = os.getenv("INDEX_NAME")
es_client = Elasticsearch('http://localhost:9200')

In [4]:
test_task_instance = TermDistribution(
    min_date="01-01-2001",
    max_date="31-12-2002",
    terms="poland,russia"
)

test_task_name = "term_distribution"

with open(f"stored_instances/{test_task_name}.pkl", "wb") as f:
    pickle.dump(test_task_instance, f)

In [33]:
query = {
    "bool": {
        "must": [
            {
                "range": {
                    "date": {
                        "gte": "2001-01-01",
                        "lte": "2002-12-31"
                    }
                }
            },
            {
                "bool": {
                    "should": [{"match": {"text": "poland"}}, {"match": {"text": "russia"}}],
                    "minimum_should_match": 1
                }
            }
        ]
    }
}
response = es_client.search(
    index=INDEX_NAME,
    query=query,
    size=10000
)

subset = response["hits"]["hits"]

In [34]:
len(subset)

312

In [6]:
rep = es_client.search(index="test", query={"match_all": {}}, size=10000)["hits"]["hits"]

In [28]:
rep[0]

{'_index': 'test',
 '_id': '1',
 '_score': 1.0,
 '_ignored': ['text.keyword'],
 '_source': {'date': '2001-08-23T00:02:00',
  'title': 'News Conference at the End of a Meeting with President Alexander Kwasniewski of Poland',
  'text': 'Good afternoon. My colleague and I would like to inform you very briefly about what we discussed today. I must say that I am very pleased with our consultations. And I want to thank the Polish President for the attention he has been personally paying to relations between Russia and Poland. I think it would be no exaggeration to say that his efforts have contributed a lot to our relations, achieving new highs in practically all areas. It is enough to say that Mr President’s visit to Russia and our repeated contacts – personally, by telephone, through correspondence – have had a positive effect above all on the economic sphere. Last year we set a 10-year record: trade between our two countries reached almost $5.5 billion.  We discussed the need to develop t